In [ ]:
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
models=['GNO', 'UNO', 'PINK', 'BROWN', 'VIOLET', # noises (R)
        'AR1_GNO', 'STAR_GNO', #    ARMA (R)
        'ARNOLD', 'CHIRIKOV', 
        'OU', 'Oscillator',# Conservative chaotic maps (R)
        'BRW_cont', 
        'HENR_diverse', 'HENR_same', 'QUADRATIC_RSUM', # Sum of frwd and bckwd realisations of chaotic maps (R)
        'AR1_UNO', 'ARMA11_UNO', 'AR3_Gamma', 'N_AR2', 'SETAR1_GNO', 'SETAR2_GNO',
        'HEN', 'HEN_SUM', 'LOGISTIC4', 'LOGISTIC38284', 'QUADRATIC', # Chaotic maps (I)
        'MODA', 'LLOG', # Other deterministic (I)
        'SINE_STOCH', # Other stochastic (I)
        'LORENZ_SUM', 'ROSSLER_SUM', 
        'MACKEYGLASS17', 'VDP', 
        'LORENZ_STOCH_SUM',
        'VDP_STOCH'
        ]

print('Models:', models)

model_keywords={ # discrete
                'GNO': 'reversible', 'UNO': 'reversible', 'PINK': 'reversible', 'BROWN': 'reversible', 'VIOLET': 'reversible',
                'AR1_GNO': 'reversible', 'STAR_GNO': 'reversible',
                'ARNOLD': 'reversible', 'CHIRIKOV': 'reversible',
                'HENR_diverse': 'reversible', 'HENR_same': 'reversible', 'QUADRATIC_RSUM': 'reversible',
                'AR1_UNO': 'irreversible', 'ARMA11_UNO': 'irreversible', 'AR3_Gamma': 'irreversible', 'N_AR2': 'irreversible', 'SETAR1_GNO': 'irreversible', 'SETAR2_GNO': 'irreversible', 
                'HEN': 'irreversible', 'HEN_SUM': 'irreversible', 'LOGISTIC4': 'irreversible', 'LOGISTIC38284': 'irreversible','QUADRATIC': 'irreversible',
                'MODA': 'irreversible', 'LLOG': 'irreversible',
                'SINE_STOCH': 'irreversible',
                # continuous
                'BRW_cont': 'reversible', 'OU': 'reversible', 'Oscillator': 'reversible',
                'LORENZ_SUM': 'irreversible', 'ROSSLER_SUM': 'irreversible',
                'MACKEYGLASS17': 'irreversible', 'VDP': 'irreversible',
                'LORENZ_STOCH_SUM': 'irreversible',
                'VDP_STOCH': 'irreversible'
        }


In [ ]:
def plot_violin_models(df, models, op, model_labels):
    
    colors={'reversible': 'blue', 'irreversible': 'red'}

    data=[]
    # Extract rows whose index is a multi iindex but that contains model
    for model in models:
        df_model = df.loc[model]
        array_op= np.array(df_model[op])
        data.append(array_op)


    fig = go.Figure()

    for i, model in enumerate(models):
        fig.add_trace(go.Violin(y=data[i], name=model, marker=dict(color=colors[model_labels[model]]), 
                                box_visible=True, meanline_visible=True, points='all', 
                                jitter=0.25, pointpos=-1.5, marker_size=2, line_width=1.2))

    fig.update_layout(title=op, width=1200, height=600,
                    plot_bgcolor='whitesmoke',  
                    paper_bgcolor='white', 
                    xaxis=dict(
                    showgrid=True,
                    tickfont=dict(size=14),
                    gridwidth=1),
                    yaxis=dict( 
                    showgrid=True,
                    tickfont=dict(size=18),
                    gridwidth=1),
                    title_font=dict(size=26),
                    showlegend=False)
    
    fig.show()

In [ ]:
cwd = os.getcwd()

dir_hctsa= cwd+'/../../../data-time-reversibility/main-analysis/data-analysis/'
dir_accuracy= cwd+'/data-analysis/accuracy/'
dir_zero=cwd+'/data-zero/'
dir_figures=cwd+'/data-analysis/figures/'
dir_supp=cwd+'/supplementary-material/'

In [ ]:
# Load good performing features
df_ops_good = pd.read_csv(dir_accuracy+'df_accuracy_good_1NN.csv')
# Load ops zero
df_ops_zero = pd.read_csv(dir_zero+'df_ops_zero.csv')

# Check what good features are among the zero ones
common_values = pd.merge(df_ops_good, df_ops_zero, left_on='Operation', right_on='Name', how='inner')
# Display the common values
print(common_values)

In [ ]:
# Load only features that are not zero
df_ops_good = df_ops_good[~df_ops_good['Operation'].isin(df_ops_zero['Name'])]
df_good_hctsa = pd.read_csv(dir_accuracy+'df_good_hctsa_1NN.csv')
df_good_hctsa.set_index('Model', inplace=True)

# Select rows with model in model
df_good_hctsa = df_good_hctsa[df_good_hctsa.index.get_level_values(0).isin(models)]

In [ ]:
# LOAD IF I WANT THE ACCURACY ON THE FULL SET OF OPERATIONS AND DATASET
# load accuracy 1NN
df_ops_full = pd.read_csv(dir_accuracy+'df_accuracy_abs_1NN.csv')

# Load full hctsa matrix
df_full_hctsa=pd.read_csv(dir_hctsa+'df_TS_DataMat_diff.csv')
df_full_hctsa.set_index(['Model'], inplace=True)

# Add column with reversible/irreversible label
df_type=df_full_hctsa.index.get_level_values(0).map(model_keywords)
df_full_hctsa.insert(0,'Type',df_type)


In [ ]:
# Load only features that are not zero
df_ops_good = df_ops_good[~df_ops_good['Operation'].isin(df_ops_zero['Name'])]

In [ ]:
op='MF_steps_ahead_ar_best_6_mabserr_1'
# MF_armax_2_2_05_1_normksstat
plot_violin_models(df_good_hctsa, models, op, model_keywords)
['SB_TransitionMatrix_51_symsumdiff', 'SB_TransitionMatrix_41_symsumdiff']

In [ ]:
# Scatter plot of the values of hctsa for the first two features in df_ops_good
# Select columns with name of features 'AC_nl_001' and 'AC_nl_002'
feat1 = 'SB_MotifTwo_diff_u'
feat2 = 'AC_nl_001'
df_two = df_ops_good[df_ops_good['Operation'].isin([feat1, feat2])]


# Select the features in df_ops_good
df_two_hctsa = df_good_hctsa[df_good_hctsa.columns[df_good_hctsa.columns.isin(df_two['Operation'])]]
# add the Type of model
df_two_hctsa['Type'] = df_two_hctsa.index.get_level_values(0).map(model_keywords)

# Scatter plot of hctsa values and color by Type
fig = px.scatter(df_two_hctsa, x=feat1, y=feat2, color='Type', 
                 title='Scatter plot of hctsa values', 
                 labels={'Type': 'Type of model', feat1: feat1, feat2: feat2},
                 color_discrete_map={'reversible': 'blue', 'irreversible': 'red'})
fig.update_traces(marker=dict(size=10, opacity=0.8), selector=dict(mode='markers'))
fig.update_layout(title=feat1 + ' vs ' + feat2, width=1200, height=600,
                plot_bgcolor='whitesmoke',  
                paper_bgcolor='white', 
                xaxis=dict(
                showgrid=True,
                tickfont=dict(size=14),
                gridwidth=1),
                yaxis=dict( 
                showgrid=True,
                tickfont=dict(size=18),
                gridwidth=1),
                title_font=dict(size=26),
                showlegend=True)
# for 1 point every 100 add name of model in the index
for i in range(0, len(df_two_hctsa), 100):
    fig.add_annotation(x=df_two_hctsa.iloc[i][feat1], y=df_two_hctsa.iloc[i][feat2], text=df_two_hctsa.index[:][i],
                     showarrow=True, arrowhead=2, ax=0, ay=-40)
fig.update_traces(textposition='top center')
fig.update_traces(textfont=dict(size=10))
fig.update_traces(textfont_color='black')
fig.update_traces(textfont_family='Arial')

fig.show()



In [ ]:
# Write to Excel with formatting
os.makedirs(dir_supp, exist_ok=True)
with pd.ExcelWriter(dir_supp+'Supplement_Table_S3.xlsx', engine='xlsxwriter') as writer:
    df_ops_good.to_excel(writer, index=False, sheet_name='Sheet1')

    workbook  = writer.book
    worksheet = writer.sheets['Sheet1']

    # Define formats
    format_grey = workbook.add_format({'bg_color': '#DDDDDD'})
    format_white = workbook.add_format({'bg_color': '#FFFFFF'})

    # Specify the header names in bold
    bold_format = workbook.add_format({'bold': True, 'font_color': '#000000', 'bg_color': '#DDDDDD'})
    header_names = ['Name', 'Global average', 'Feature class', 'Accuracy']
    # Write the header with bold format
    for col_num, header in enumerate(df_ops_good.columns):
        worksheet.write(0, col_num, header, bold_format)


    # Apply alternating colors row by row
    for row in range(1, len(df_ops_good)+1):  # Start at 1 to skip header
        row_format = format_grey if row % 2 == 0 else format_white
        worksheet.set_row(row, 20, row_format)
    worksheet.set_column('A:A', 40)  # Set width for the first column
    worksheet.set_column('B:B', 20)  # Set width for the second column
    worksheet.set_column('C:C', 20)  # Set width for the third column
